### This is to be used with Colab (A100 40-GB)

In [2]:
! nvidia-smi

Sun Mar  1 12:30:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:01:00.0 Off |                  N/A |
|  0%   38C    P8             27W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!apt-get -qq update
!apt-get -qq install -y openjdk-21-jre-headless libomp-dev

# Force java to 21 (important)
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
!java -version

!pip -q install -U huggingface_hub hf_transfer transformers pyserini

# FAISS GPU for CUDA 12.x
!pip -q install -U faiss-gpu-cu12


E: Could not open lock file /var/lib/apt/lists/lock - open (13: Permission denied)
E: Unable to lock directory /var/lib/apt/lists/
W: Problem unlinking the file /var/cache/apt/pkgcache.bin - RemoveCaches (13: Permission denied)
W: Problem unlinking the file /var/cache/apt/srcpkgcache.bin - RemoveCaches (13: Permission denied)
E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-124.04, mixed mode, sharing)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
adapters 1.2.0 requires transformers~=4.51.3, but you have transformers 5.2.0 which is incompatible.


In [3]:
# ============================================================
# Hybrid Retrieval (OPTIMIZED)
# BM25 + DPR FAISS + BGE Reranker
# ============================================================

import os
import subprocess

# -----------------------------
# Force Java 21 BEFORE pyserini
# -----------------------------
JAVA_HOME = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["JDK_HOME"] = JAVA_HOME
os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = os.path.join(JAVA_HOME, "lib", "server") + os.pathsep + os.environ.get("LD_LIBRARY_PATH", "")

print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())

# -----------------------------
# Imports
# -----------------------------
import json
import numpy as np
import faiss
import torch
import gc

from huggingface_hub import snapshot_download, hf_hub_download
from pyserini.search.lucene import LuceneSearcher

from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizerFast,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# ============================================================
# CONFIG (OPTIMIZED)
# ============================================================

DPR_REPO  = "MinaGabriel/wikipedia-18-en-dpr-faiss-100tok"
BM25_REPO = "MinaGabriel/wiki18-bm25-index"

DPR_MODEL         = "facebook/dpr-question_encoder-single-nq-base"
RERANK_MODEL_NAME = "BAAI/bge-reranker-base"

# 🔥 Reduced safely
K_BM25  = 100
K_DENSE = 30
FUSE_K  = 100     # ↓ from 200 (no real loss)
TOP_K   = 30

# 🔥 Major RAM saver (no impact on reranker)
DPR_TEXT_CHARS  = 512
BM25_TEXT_CHARS = 512

DENSE_GPU  = 0
RERANK_GPU = 0

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

# ============================================================
# DEVICES
# ============================================================
use_cuda = torch.cuda.is_available()
device_dense  = torch.device(f"cuda:{DENSE_GPU}" if use_cuda else "cpu")
device_rerank = torch.device(f"cuda:{RERANK_GPU}" if use_cuda else "cpu")

print(f"Dense device: {device_dense}")
print(f"Reranker device: {device_rerank}")

# ============================================================
# DOWNLOAD INDEX
# ============================================================
print("Downloading FAISS + META...")

INDEX_PATH = hf_hub_download(
    repo_id=DPR_REPO,
    repo_type="dataset",
    filename="index/wiki_dpr.index",
)

META_PATH = hf_hub_download(
    repo_id=DPR_REPO,
    repo_type="dataset",
    filename="index/wiki_dpr_meta.npz",
)

# ============================================================
# LOAD BM25
# ============================================================
print("Loading BM25...")
bm25_dir = snapshot_download(repo_id=BM25_REPO, repo_type="dataset")
searcher = LuceneSearcher(os.path.join(bm25_dir, "bm25_index"))

# ============================================================
# LOAD FAISS
# ============================================================
print("Loading FAISS index...")
index_cpu = faiss.read_index(INDEX_PATH)

if use_cuda:
    try:
        print("Moving FAISS to GPU...")
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, DENSE_GPU, index_cpu)
        del index_cpu
        gc.collect()
        print("FAISS on GPU ✅")
    except Exception as e:
        print("Fallback CPU FAISS:", e)
        index = index_cpu
else:
    index = index_cpu

# ============================================================
# LOAD META (🔥 MEMORY-MAPPED)
# ============================================================
print("Loading metadata (mmap)...")

meta = np.load(META_PATH, allow_pickle=True, mmap_mode="r")

doc_ids = meta["doc_ids"]
titles  = meta["titles"]
texts   = meta["texts"]   # lazy-loaded now

print(f"Docs: {len(doc_ids):,}")

# ============================================================
# LOAD MODELS
# ============================================================
print("Loading DPR...")
q_tok = DPRQuestionEncoderTokenizerFast.from_pretrained(DPR_MODEL)
q_model = DPRQuestionEncoder.from_pretrained(DPR_MODEL).to(device_dense).eval()

print("Loading BGE reranker...")
rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME)
rerank_model = AutoModelForSequenceClassification.from_pretrained(
    RERANK_MODEL_NAME,
    torch_dtype=torch.float16 if device_rerank.type == "cuda" else torch.float32,
).to(device_rerank).eval()

# ============================================================
# HELPERS
# ============================================================
def _norm_key(title, text):
    return (title or "").lower() + "||" + (text or "")[:200].lower()

# ============================================================
# DPR SEARCH
# ============================================================
@torch.no_grad()
def dense_search(question, top_k=30):
    enc = q_tok(question, return_tensors="pt", truncation=True, max_length=64)
    enc = {k: v.to(device_dense) for k, v in enc.items()}

    emb = q_model(**enc).pooler_output
    vec = emb.detach().cpu().numpy().astype(np.float32)
    faiss.normalize_L2(vec)

    scores, idxs = index.search(vec, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), start=1):
        if idx < 0:
            continue

        txt = texts[idx]
        txt = txt[:DPR_TEXT_CHARS] if txt else ""

        results.append({
            "source": "dpr",
            "rank": rank,
            "score": float(score),
            "doc_id": str(doc_ids[idx]),
            "title": str(titles[idx]),
            "text": txt,
        })

    return results

# ============================================================
# BM25 SEARCH
# ============================================================
def bm25_search(question, top_k=100):
    hits = searcher.search(question, top_k)

    results = []
    for rank, h in enumerate(hits, start=1):
        record = json.loads(searcher.doc(h.docid).raw())

        txt = record.get("contents", "")
        txt = txt[:BM25_TEXT_CHARS] if txt else ""

        results.append({
            "source": "bm25",
            "rank": rank,
            "score": float(h.score),
            "doc_id": str(h.docid),
            "title": record.get("title", ""),
            "text": txt,
        })

    return results

# ============================================================
# RRF FUSION
# ============================================================
def fuse_candidates(bm25_res, dpr_res, fuse_k=100, rrf_k=60):
    pool = {}

    for r in (bm25_res or []) + (dpr_res or []):
        key = _norm_key(r["title"], r["text"])

        if key not in pool:
            pool[key] = {**r, "rrf": 0.0}

        pool[key]["rrf"] += 1.0 / (rrf_k + r["rank"])

    fused = sorted(pool.values(), key=lambda x: x["rrf"], reverse=True)
    return fused[:fuse_k]

# ============================================================
# BGE RERANK
# ============================================================
@torch.no_grad()
def rerank_with_bge(question, candidates, top_k=30, batch_size=8):
    if not candidates:
        return []

    pairs = [(question, c["text"]) for c in candidates]
    scores = np.zeros(len(pairs), dtype=np.float32)

    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]

        enc = rerank_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=384,
            return_tensors="pt"
        )
        enc = {k: v.to(device_rerank) for k, v in enc.items()}

        logits = rerank_model(**enc).logits.squeeze(-1)
        scores[i:i+len(batch)] = logits.detach().cpu().numpy()

    order = np.argsort(-scores)[:top_k]

    return [
        {**candidates[i], "rerank_score": float(scores[i])}
        for i in order
    ]

# ============================================================
# FINAL PIPELINE
# ============================================================
def retrieve_and_rerank(question):
    bm25 = bm25_search(question)
    dpr  = dense_search(question)

    fused = fuse_candidates(bm25, dpr)
    reranked = rerank_with_bge(question, fused)

    return reranked

openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-124.04, mixed mode, sharing)



Dense device: cuda:0
Reranker device: cuda:0
Loading BM25...


Fetching 190 files:   0%|          | 0/190 [00:00<?, ?it/s]

Mar 01, 2026 12:31:00 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


Loading FAISS index...
Moving FAISS to GPU...
FAISS on GPU ✅
Loading metadata (mmap)...


: 

In [1]:
# ============================================================
# SMOKE TEST
# ============================================================
q = "Where was Luke Prokopec born?"
out = retrieve_and_rerank(q, top_k=20) 
print(json.dumps(out[0], indent=2))
for r in out:
    print(r["source"], f"{r.get('rerank_score', float('nan')):.4f}", r["text"][:250].replace("\n", " "))


NameError: name 'retrieve_and_rerank' is not defined

In [ ]:
# ============================================================
# SIMPLE GRANOLA RETRIEVAL PIPELINE (CLEAN)
# ============================================================

import os, json, shutil
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset

# -------------------------
# SETTINGS
# -------------------------
DATASET_NAME = "google/granola-entity-questions"
TOP_K = 20
K_LIST = [1, 5, 10, 20]

RUN_DIR = "granola_outputs"
SAVE_DIR = "dataset_with_retrieval"

# -------------------------
# SIMPLE NORMALIZATION
# -------------------------
def norm(s):
    return "" if s is None else str(s).lower()

def hit_rank(retrieved_docs, answer):
    answer = norm(answer)
    for i, d in enumerate(retrieved_docs[:TOP_K], start=1):
        text = norm(d.get("text", ""))
        if answer and answer in text:
            return i
    return None

def mean(xs):
    return float(np.mean(xs)) if xs else 0.0

# -------------------------
# LOAD DATASET
# -------------------------
ds = load_dataset(DATASET_NAME, split="test")

print("Dataset loaded:", len(ds))

# -------------------------
# MAIN LOOP
# -------------------------
all_hits = {k: [] for k in K_LIST}
rows = []

for ex in tqdm(ds, desc="Retrieving"):
    question = ex["question"]
    answer = ex["answer"]   # HARD CODED

    # 🔴 YOUR RETRIEVAL FUNCTION
    retrieved = retrieve_and_rerank(
        question,
        k_bm25=500,
        k_dense=500,
        fuse_k=500,
        top_k=TOP_K,
    )

    rank = hit_rank(retrieved, answer)

    # metrics
    for k in K_LIST:
        all_hits[k].append(1 if (rank is not None and rank <= k) else 0)

    # store row
    row = dict(ex)
    row["retrieved_docs"] = retrieved
    row["hit_rank"] = rank if rank else -1

    rows.append(row)

# -------------------------
# REPORT
# -------------------------
os.makedirs(RUN_DIR, exist_ok=True)

report = []
report.append("Granola Retrieval Report")
report.append("=" * 50)

for k in K_LIST:
    score = mean(all_hits[k])
    report.append(f"R@{k}: {score:.4f}")

report_text = "\n".join(report)

with open(os.path.join(RUN_DIR, "report.txt"), "w") as f:
    f.write(report_text)

print(report_text)

# -------------------------
# SAVE DATASET
# -------------------------
dataset = Dataset.from_list(rows)

save_path = os.path.join(RUN_DIR, SAVE_DIR)

if os.path.exists(save_path):
    shutil.rmtree(save_path)

dataset.save_to_disk(save_path)

print("Saved dataset to:", save_path)

In [ ]:
! cat granola_run_outputs/granola_entity_questions_retrieval_rank_report.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 